# ANÁLISE DE MÉTRICAS - SPOTIFY
Samuel Pimenta, Vinícius Vilas Boas, Isabelly da Hora, Caio Mezini, Pedro Casarini 2°G

## Escolha da base de dados
Escolhemos uma base de dados do Spotify, que contém informações sobre músicas, como nome, artista, gênero, popularidade, duração, entre outros. A base de dados foi obtida através do Kaggle e possui mais de 100 mil registros de músicas.

[Spotify Tracks Genre Dataset](https://www.kaggle.com/datasets/thedevastator/spotify-tracks-genre-dataset)

## Colunas da base de dados
- artists: Nome(s) do(s) artista(s) associado(s) à faixa. (String)
- album_name: O nome do álbum ao qual a faixa pertence. (String)
- track_name: O nome da faixa. (String)
- popularity: A pontuação de popularidade da faixa no Spotify, variando de 0 a 100. (Integer)
- duration_ms: A duração da faixa em milissegundos. (Integer)
- explicit: Um valor booleano que indica se a faixa contém conteúdo explícito. (Boolean)
- danceability: Uma pontuação de 0 a 1 que representa o quão adequada uma faixa é para dançar com base em vários elementos musicais. (Float)
- energy: Uma medida da intensidade e atividade de uma faixa, variando de 0 a 1. (Float)
- key: A chave da faixa representada por um valor inteiro. (Integer)
- loudness: Os decibels da faixa (dB). (Float)
- mode: O modo tonal da faixa, representado por um valor inteiro (0 para menor, 1 para maior). (Integer)
- speechiness: Uma pontuação que varia de 0 a 1 e representa a presença de palavras faladas em uma faixa. (Float)
- acousticness: Uma pontuação que varia de 0 a 1 e representa o grau em que uma faixa possui uma qualidade acústica. (Float)
- instrumentalness: Uma pontuação que varia de 0 a 1 e representa a probabilidade de uma faixa ser instrumental. (Float)
- liveness: Uma pontuação que varia de 0 a 1 e representa a presença de um público durante a gravação ou performance de uma faixa. (Float)
- valence: Uma pontuação que varia de 0 a 1 e representa a positividade musical transmitida por uma faixa. (Float)
- tempo: O tempo da faixa em batidas por minuto (BPM). (Float)
- time_signature: O número de batidas dentro de cada barra da faixa. (Integer)
- track_genre: O gênero da faixa. (String)

## Tratamento e preparação dos dados

In [ ]:
import pandas as pd

df = spark.table('workspace.trabalho_bi.spotify_genre')
df_pandas = df.toPandas()

In [ ]:
df_pandas.head()

## Mudando nomes das colunas

In [ ]:
df_pandas = df_pandas.rename(
    columns={
        'Unnamed: 0':'nao_nomeado',
        'track_id':'id_musica',
        'artists':'artistas',
        'album_name':'nome_album',
        'track_name':'nome_musica',
        'popularity':'popularidade',
        'duration_ms':'duracao_ms',
        'explicit':'explicito',
        'danceability':'danceabilidade',
        'energy':'energia',
        'key':'tonalidade',
        'loudness':'nivel_barulho',
        'mode':'modo',
        'speechiness':'fala',
        'acousticness':'acustico',
        'instrumentalness':'instrumental',
        'liveness':'vivacidade',
        'valence':'valencia',
        'tempo':'tempo',
        'time_signature':'assinatura_tempo',
        'track_genre':'genero'
    }
)

In [ ]:
df_pandas.head()

In [ ]:
df_pandas.dtypes

## Contagem de valores nulos

In [ ]:
columns = ['nao_nomeado', 'id_musica', 'artistas', 'nome_album', 'nome_musica', 'popularidade', 'duracao_ms', 'explicito', 'danceabilidade', 'energia', 'tonalidade', 'nivel_barulho', 'modo', 'fala', 'acustico', 'instrumental', 'vivacidade', 'valencia', 'tempo', 'assinatura_tempo', 'genero']

nulos = df_pandas[columns].isnull().sum().reset_index()
nulos.columns = ['coluna', 'qnt_nulos']
nulos['pct_nulos'] = (nulos['qnt_nulos'] / len(df_pandas) * 100).round(2)

print(f"Total de linhas: {len(df_pandas)}\n")
print(nulos.to_string(index=False))

In [ ]:
colunas_verificar = ['artistas', 'nome_album', 'nome_musica']

df_nulos = df_pandas[df_pandas[colunas_verificar].isnull().any(axis=1)]

print(df_nulos)

In [ ]:
linha = df_pandas.iloc[65900]

for coluna, valor in linha.items():
    print(f"{coluna}: {valor}")

In [ ]:
df_pandas = df_pandas.drop(index=65900)

In [ ]:
nulos = df_pandas[columns].isnull().sum().reset_index()
nulos.columns = ['coluna', 'qnt_nulos']
nulos['pct_nulos'] = (nulos['qnt_nulos'] / len(df_pandas) * 100).round(2)

print(f"Total de linhas: {len(df_pandas)}\n")
print(nulos.to_string(index=False))

## Tratamento de valores nulos

In [ ]:
import random
import string

# ---- nao_nomeado: sequência crescente sem duplicar ----
ids_existentes = set(df['nao_nomeado'].dropna().astype(int).tolist())

def gerar_nao_nomeado():
    proximo = max(ids_existentes) + 1 if ids_existentes else 1
    while proximo in ids_existentes:
        proximo += 1
    ids_existentes.add(proximo)
    return proximo

mask_nao_nomeado = df_pandas['nao_nomeado'].isnull()
df_pandas.loc[mask_nao_nomeado, 'nao_nomeado'] = [gerar_nao_nomeado() for _ in range(mask_nao_nomeado.sum())]
df_pandas['nao_nomeado'] = df_pandas['nao_nomeado'].astype('int64')

# ---- id_musica: id aleatório único ----
ids_musica_existentes = set(df_pandas['id_musica'].dropna().tolist())

def gerar_id_musica(tamanho=22):
    caracteres = string.ascii_letters + string.digits
    while True:
        novo_id = ''.join(random.choices(caracteres, k=tamanho))
        if novo_id not in ids_musica_existentes:
            ids_musica_existentes.add(novo_id)
            return novo_id

mask_id_musica = df_pandas['id_musica'].isnull()
df_pandas.loc[mask_id_musica, 'id_musica'] = [gerar_id_musica() for _ in range(mask_id_musica.sum())]

# ---- demais colunas ----
padronizacao = {
    'artistas'        : 'Não Informado',
    'nome_album'      : 'Não Informado',
    'nome_musica'     : 'Não Informado',
    'popularidade'    : 0,
    'duracao_ms'      : 197000,
    'explicito'       : False,
    'danceabilidade'  : 0.5,
    'energia'         : 0.5,
    'tonalidade'      : 5,
    'nivel_barulho'   : -22.485,
    'modo'            : 1,
    'fala'            : 0.3,
    'acustico'        : 0.5,
    'instrumental'    : 0.01,
    'vivacidade'      : 0.3,
    'valencia'        : 0.5,
    'tempo'           : 121.0,
    'assinatura_tempo': 4,
    'genero'          : 'Não Informado'
}

for coluna, valor_padrao in padronizacao.items():
    df_pandas[coluna] = df_pandas[coluna].fillna(valor_padrao)

# ---- garantir tipos corretos ----
tipos = {
    'popularidade'    : 'int64',
    'duracao_ms'      : 'int64',
    'explicito'       : 'bool',
    'danceabilidade'  : 'float64',
    'energia'         : 'float64',
    'tonalidade'      : 'int64',
    'nivel_barulho'   : 'float64',
    'modo'            : 'int64',
    'fala'            : 'float64',
    'acustico'        : 'float64',
    'instrumental'    : 'float64',
    'vivacidade'      : 'float64',
    'valencia'        : 'float64',
    'tempo'           : 'float64',
    'assinatura_tempo': 'int64'
}

for coluna, tipo in tipos.items():
    df_pandas[coluna] = df_pandas[coluna].astype(tipo)

print("✅ Padronização concluída!")
print(f"Total de linhas: {len(df_pandas)}")
print(f"\nNulos restantes:\n{df_pandas.isnull().sum()[df_pandas.isnull().sum() > 0]}")

## Verificar tipos de dados

### Colunas Object

In [ ]:
# Verifica colunas object que têm valores não-string
colunas_object = ['id_musica', 'artistas', 'nome_album', 'nome_musica', 'genero']

print(f"{'COLUNA':<20} {'LINHAS COM TIPO ERRADO'}")
print("-" * 45)

for coluna in colunas_object:
    mask = df_pandas[coluna].apply(lambda x: not isinstance(x, str) and pd.notna(x))
    qtd = mask.sum()
    status = f"{qtd} linha(s)" if qtd > 0 else "✅ OK"
    print(f"{coluna:<20} {status}")

    if qtd > 0:
        print(df_pandas[mask][['nao_nomeado', coluna]].to_string(index=False))
        print()

### Colunas Float

In [ ]:
import numpy as np

colunas_float = ['danceabilidade', 'energia', 'nivel_barulho', 'fala', 'acustico', 'instrumental', 'vivacidade', 'valencia', 'tempo']

print(f"{'COLUNA':<20} {'LINHAS COM TIPO ERRADO'}")
print("-" * 45)

for coluna in colunas_float:
    mask = df_pandas[coluna].apply(lambda x: not isinstance(x, (float, np.floating)) and pd.notna(x))
    qtd = mask.sum()
    status = f"{qtd} linha(s)" if qtd > 0 else "✅ OK"
    print(f"{coluna:<20} {status}")

    if qtd > 0:
        print(df_pandas[mask][['nao_nomeado', coluna]].to_string(index=False))
        print()

### Colunas Int

In [ ]:
import numpy as np

colunas_int = ['nao_nomeado', 'popularidade', 'duracao_ms',
               'tonalidade', 'modo', 'assinatura_tempo']

print(f"{'COLUNA':<20} {'LINHAS COM TIPO ERRADO'}")
print("-" * 45)

for coluna in colunas_int:
    mask = df_pandas[coluna].apply(lambda x: not isinstance(x, (int, np.integer)) and pd.notna(x))
    qtd = mask.sum()
    status = f"{qtd} linha(s)" if qtd > 0 else "✅ OK"
    print(f"{coluna:<20} {status}")

    if qtd > 0:
        print(df_pandas[mask][['nao_nomeado', coluna]].to_string(index=False))
        print()

### Coluna Bool

In [ ]:
colunas_bool = ['explicito']

print(f"{'COLUNA':<20} {'LINHAS COM TIPO ERRADO'}")
print("-" * 45)

for coluna in colunas_bool:
    mask = df_pandas[coluna].apply(lambda x: not isinstance(x, bool) and pd.notna(x))
    qtd = mask.sum()
    status = f"{qtd} linha(s)" if qtd > 0 else "✅ OK"
    print(f"{coluna:<20} {status}")

    if qtd > 0:
        print(df_pandas[mask][['nao_nomeado', coluna]].to_string(index=False))
        print()

## Verificação de registros duplicados

In [ ]:
duplicados = df_pandas[df_pandas.duplicated(subset=['id_musica'], keep=False)]

print(f"Total de registros: {len(df_pandas)}")
print(f"Total de linhas duplicadas: {len(duplicados)}")
print(f"Total de id_musica duplicados: {df_pandas['id_musica'].duplicated().sum()}\n")

if len(duplicados) > 0:
    resultado = duplicados[['nao_nomeado', 'id_musica', 'nome_musica', 'artistas']]\
        .sort_values('id_musica')\
        .to_json(orient='records', indent=4)

    print(resultado)